# MUSE — all experiments on Kaggle (2 × T4, free)

Covers **both** parts of the paper: brain-tumour MRI (3D U-Net MUSE vs baselines) and the graph benchmarks (Cora, CiteSeer, Amazon-Photo).
Choose what a run does with `RUN` in the first code cell (each Kaggle run lasts at most 12 h).

**Before the first run (once):**
1. Right panel → *Session options*: **Accelerator = GPU T4 x2**, **Internet = On** (needs a phone-verified Kaggle account).
2. *Optional* — top menu *Add-ons → Secrets*: add **`GH_TOKEN`** (a GitHub token) and tick it for this notebook.
   Without it the run still works; results are not pushed and stay in the notebook output (`runs/*/test.json`).

**To run:** click **Save Version → Save & Run All (Commit)**. It runs by itself for up to 12 h — you can close the laptop.
With a token, every finished model is pushed to GitHub as a results PR.

**To continue after a 12 h run:** *Add Input → Your Work → Notebooks →* select this notebook's latest version
(its output holds the checkpoints), then **Save & Run All** again. Finished models are skipped and unfinished ones resume.

In [ ]:
# 0. What to run in this session: ['mri'], ['graph'] or ['mri', 'graph']
RUN = ['mri']
# Seed-robustness run: MUSE and the two strongest baselines, two extra seeds
# (seed 0 of every model is already in results/mri3d_runs)
MODELS = 'MUSE,mmFormer,RFNet'
SEEDS = '1,2'

# 1. GPU, secrets, code
import os, glob, shutil, subprocess
!nvidia-smi --query-gpu=name,memory.total --format=csv
from kaggle_secrets import UserSecretsClient
import json, urllib.request
try:  # the token is optional: without it nothing is pushed and results stay in the output
    os.environ['GH_TOKEN'] = UserSecretsClient().get_secret('GH_TOKEN').strip()
    req = urllib.request.Request('https://api.github.com/repos/Kxvya18/MUSE', headers={'Authorization': 'Bearer ' + os.environ['GH_TOKEN']})
    print('GitHub token OK:', json.load(urllib.request.urlopen(req))['full_name'])
except Exception as e:
    os.environ.pop('GH_TOKEN', None)
    print(f'No usable GH_TOKEN ({e}): results will only be saved in the notebook output.')
os.chdir('/kaggle/working')
if not os.path.exists('MUSE'):
    !git clone -q https://github.com/Kxvya18/MUSE.git  # public repo: no token on disk
os.chdir('/kaggle/working/MUSE')
!git pull -q
!git config user.name "Kxvya18" && git config user.email "174275647+Kxvya18@users.noreply.github.com"
!pip -q install nibabel torch_geometric

In [ ]:
# 2. Resume: bring back checkpoints from a previous version attached as input
RUNS = '/kaggle/working/runs'
os.makedirs(RUNS, exist_ok=True)
for prev in glob.glob('/kaggle/input/*/runs'):
    print('resuming from', prev)
    !cp -rn {prev}/. {RUNS}/
!ls {RUNS}

In [ ]:
if 'mri' in RUN:
    # 3. Data: download (~7.6 GB) + preprocess (~20 min) to local scratch
    if not os.path.exists('/tmp/mri3d/split.json'):
        !wget -q -O /tmp/t.tar https://msd-for-monai.s3-us-west-2.amazonaws.com/Task01_BrainTumour.tar
        !tar -xf /tmp/t.tar -C /tmp && rm /tmp/t.tar
        !python run_mri3d.py prep --src /tmp/Task01_BrainTumour --data /tmp/mri3d --workers 4
    !ls /tmp/mri3d | wc -l

In [ ]:
# 4. MRI: train + test, two models at a time (one per GPU). Progress is printed every 5 min
#    and published to https://github.com/Kxvya18/MUSE/blob/kaggle-progress/PROGRESS.md
if 'mri' in RUN:
    !python kaggle/run_queue.py --runs {RUNS} --data /tmp/mri3d --models {MODELS} --seeds {SEEDS} --extra "--iters 12000 --patch 96 --test_patch 128 --test_stride 96"
    !tail -n 3 {RUNS}/log_*.txt
    # small archive (test/train JSON + logs, no weights) for download from the Output tab
    !cd {RUNS} && tar czf /kaggle/working/mri_seed_results.tgz $(ls -d */test.json */train.json log_*.txt 2>/dev/null)

## 5. Graph benchmarks: validation tuning → config selection → 5-seed test runs → ablations → fusion-order study
Finished runs are kept in the notebook output, so a later run resumes them.

In [ ]:
if 'graph' in RUN:
    G = '/kaggle/working/graph_results'
    os.makedirs(G, exist_ok=True)
    for prev in glob.glob('/kaggle/input/*/graph_results'):
        !cp -rn {prev}/. {G}/
    !cp -rn results/. {G}/
    # results of models whose code changed since the previous run are recomputed
    !rm -f {G}/tune/*_EGNN_* {G}/final/*_EGNN_*
    MAIN = 'MUSE,GCN-zero,GAT-zero,APPNP-zero,MMGL-style,TMC,ECD,EGNN,HeMIS'
    ABL = 'MUSE-noProp,MUSE-noAux,MUSE-selfEvidence,MUSE-induced,MUSE-noGate,MUSE-Dempster,MUSE-zeroimpute,MUSE-specVar,MUSE-noVar,MUSE-MLP'
    # run_graph.py writes to ./results, so point it at the persistent copy
    shutil.rmtree('results'); os.symlink(G, 'results')
    !python run_graph.py tune --datasets Cora,CiteSeer --models {MAIN} --grid small
    !python run_graph.py tune --datasets Photo --models {MAIN} --grid photo --seeds 0
    !python run_graph.py select
    !python run_graph.py final --datasets Cora,CiteSeer,Photo --models {MAIN}
    !python run_graph.py final --datasets Cora,CiteSeer --models {ABL}
    !python analysis_order.py
    # one small archive with every result (JSON only) for easy download
    !tar czf /kaggle/working/graph_results.tgz -C {G} --exclude='*.pt' .
    # restore the real results/ directory and push everything as one PR
    os.unlink('results')
    !git checkout -q -- results
    !cp -rn {G}/. results/
    !python scripts/colab_pr.py results/kaggle-graph-$(date +%Y%m%d-%H%M) "Graph benchmark results (Kaggle)" "Validation tuning, selected configs, 5-seed test results, ablations and fusion-order study for Cora/CiteSeer/Photo." results